# 2. Generating Ground Truth Data

In [1]:
%load_ext autoreload
%autoreload 2
import dotenv

dotenv.load_dotenv(override=True)

True

In [2]:
from src import FaqHttpLoader

loader = FaqHttpLoader()
documents = loader.load()

In [3]:
print(documents[0]['id'])
print(documents[0]['question'])

0e38656cfb
How do I submit homework?


Generating questions with structured output

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.
""".strip()

In [5]:
from openai import OpenAI

ollama_client = OpenAI(
    api_key='ollama',
    base_url='http://localhost:11434/v1',
)

def llm_structured(
    instructions,
    user_prompt,
    output_type,
    model='granite4.1:8b'
    ):
    messages = [
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=output_type,
        max_tokens=1024,
    )

    return response.choices[0].message.parsed

In [6]:
import json

result = llm_structured(
    data_gen_instructions,
    json.dumps(documents[0]),
    Questions
)

print(result.questions)

['What is the process for completing and submitting homework assignments in the Machine Learning ZoomCamp course?', 'Where should I host my completed homework code before submission?', 'How can I ensure that my submitted answers are visible after the deadline has passed?', 'In which directory of the GitHub repository are the homework materials located for the 2025 cohort?', 'Through which platform do I need to submit my homework?']


Parallel processing

In [7]:
import json
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

CACHE_DIR = Path('../../data/ground_truth')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results


def process(doc):
    cache_file = CACHE_DIR / f"{doc['id']}.json"

    if cache_file.exists():
        return json.loads(cache_file.read_text())

    out = llm_structured(
        data_gen_instructions,
        json.dumps(doc),
        Questions
    )

    results = [
        {'question': q, 'course': doc['course'], 'document': doc['id']}
        for q in out.questions
    ]

    cache_file.write_text(json.dumps(results, ensure_ascii=False, indent=2))
    return results

Generate questions for all documents:

In [9]:
with ThreadPoolExecutor(max_workers=6) as pool:
    ground_truth = map_progress(pool, documents, process)

  0%|          | 0/1208 [00:00<?, ?it/s]

Flatten the nested lists into a single dataset:

In [10]:
import pandas as pd

ground_truth_flat = [item for sublist in ground_truth for item in sublist]
df_ground_truth = pd.DataFrame(ground_truth_flat)

print(len(df_ground_truth))

6020


In [11]:
# Save it for later use:
df_ground_truth.to_csv(Path('../../data')/'ground-truth-data.csv', index=False)


# 3. Search Evaluation

Let's set up our search using RAGBase from module 01:

In [12]:
from src import FaqHttpLoader, MinsearchIndex, RAGBase, OllamaClient

# Load documents
loader = FaqHttpLoader()
documents = loader.load()

# Index
index = MinsearchIndex(documents)

# LLM-client (Ollama, local)
llm_client = OllamaClient()

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=llm_client,
    llm_model='granite4.1:8b',
    instructions=instructions,
)

We'll use assistant.search to evaluate different boost configurations.

In [16]:
def search_fn(query, course):
    return assistant.search(
        query,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': course},
    )

Collecting relevance data

In [25]:
relevance_total = []

for q in tqdm(ground_truth_flat):
    doc_id = q['document']
    results = search_fn(query=q['question'], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

  0%|          | 0/6020 [00:00<?, ?it/s]

**Hit Rate**

Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results:

$$\text{Hit Rate} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \mathbb{1}(q_i \in R(q_i))$$

Where $Q$ is the set of queries, $R(q_i)$ is the set of retrieved documents for query $q_i$, and $\mathbb{1}$ is the indicator function.

In [32]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

hit_rate_value = hit_rate(relevance_total)
print(f"Hit Rate (Recall@k): {hit_rate_value:.3f}")

Hit Rate (Recall@k): 0.770


**Mean Reciprocal Rank (MRR)**

In [33]:
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)


mrr_value = mrr(relevance_total)
print(f"MRR: {mrr_value:.3f}")

MRR: 0.632


Putting it together

In [36]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }



In [37]:
evaluate(
    ground_truth_flat,
    lambda q: search_fn(q['question'], q['course'])
)

  0%|          | 0/6020 [00:00<?, ?it/s]

{'hit_rate': 0.770265780730897, 'mrr': 0.63217331118494}

Try different boost values to see what works best:

In [39]:
def search_boost(query, course, boost_val):
    return assistant.search(
        query,
        boost_dict={'question': 1.0, "answer": boost_val, 'section': 0.5},
        filter_dict={'course': course},
    )
{"question": 3, "answer": 1, "section": 0.5}
for boost in [1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth_flat,
        lambda q: search_boost(q['question'], q['course'], boost)
    )
    print(f'boost={boost}: {result}')

  0%|          | 0/6020 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.838704318936877, 'mrr': 0.7145044296788488}


  0%|          | 0/6020 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8906976744186047, 'mrr': 0.7798172757475076}


  0%|          | 0/6020 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8813953488372093, 'mrr': 0.7715005537098552}


  0%|          | 0/6020 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8654485049833887, 'mrr': 0.7551135105204867}


# 4. RAG Evaluation: Cosine Similarity

**Generating RAG answers**

In [ ]:
from src import FaqHttpLoader, MinsearchIndex, RAGBase, OllamaClient

documents = FaqHttpLoader().load()
index = MinsearchIndex(documents)
llm_client = OllamaClient()

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=llm_client,
    llm_model='granite4.1:3b',
    instructions=instructions,
)

Now run RAG on all ground truth questions and collect both the LLM answer and the original answer:

In [ ]:
doc_idx = {d['id']: d for d in documents}

answers = {}
for i, rec in enumerate(tqdm(ground_truth_flat)):
    if i in answers:
        continue
    answer_llm = assistant.rag(
        rec['question'],
        filter_dict={'course': rec['course']}
    )
    doc_id = rec['document']
    original_doc = doc_idx[doc_id]
    answer_orig = original_doc['answer']

    answers[i] = {
        'answer_llm': answer_llm,
        'answer_orig': answer_orig,
        'document': doc_id,
        'question': rec['question'],
        'course': rec['course'],
    }

    

  0%|          | 0/6020 [00:00<?, ?it/s]

In [50]:
answer_llm

'To submit homework assignments in the machine-learning-zoomcamp course, follow these steps based on the provided context:\n\n1. **Access the Homework Form**: Log into the course platform when the homework submission form opens. Enrollment is not required for submitting homework; the Airtable registration is only for announcements.\n\n2. **Submit Your Work via GitHub**: Follow the guidelines specified in the homework instructions to submit your assignments through your GitHub repository as indicated.\n\n3. **Include Public Links (Optional)**: If you wish to share what you learned publicly, use the designated section in the submission form to enter public links. Separate multiple links with any whitespace character (e.g., linebreak, space, tab). Note that:\n   - You earn extra scores for posting learning links using the `#mlzoomcamp` tag.\n   - The maximum number of points for public links is 7. If you provide more than 7 links, only 7 points will be awarded.\n   - For weekly posts acro